# Klasyfikacja Irysów

In [19]:
# imports
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import accuracy_score

from src.decision_table import DecisionTable

### 1. Dane

In [20]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['id'] = range(len(df))

### 2. Podział 70% trening, 30% test

In [21]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

### 3. Dyskretyzacja

In [22]:
est = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
cols = iris.feature_names

train_df_disc = train_df.copy()
test_df_disc = test_df.copy()

train_df_disc[cols] = est.fit_transform(train_df[cols])
test_df_disc[cols] = est.transform(test_df[cols])

mapping = {0.0: 'Low', 1.0: 'Medium', 2.0: 'High'}
for c in cols:
    train_df_disc[c] = train_df_disc[c].map(mapping)
    test_df_disc[c] = test_df_disc[c].map(mapping)

### 4. Aproksymacje

In [23]:
dt = DecisionTable(train_df_disc, universe_column='id', decision_attribute='target')

print("Aproksymacje")
for val in dt.get_decision_values():
    class_name = iris.target_names[val]
    subset = dt.get_objects_with_decision_value(val)
    
    lower = dt.get_lower_approximation(subset)
    upper = dt.get_upper_approximation(subset)
    boundary = dt.get_boundary_region(subset)
    
    print(f"Klasa {class_name:10}: Dolna: {len(lower):2}, Górna: {len(upper):2}, Granica: {len(boundary)}")

Aproksymacje
Klasa versicolor: Dolna: 32, Górna: 55, Granica: 23
Klasa virginica : Dolna: 19, Górna: 42, Granica: 23
Klasa setosa    : Dolna: 31, Górna: 31, Granica: 0


### 5. Redukt

In [24]:
reducts = dt.get_attribute_reducts()
chosen_reduct = reducts[0] if reducts else dt.get_conditionals()
print(f"\n--- PUNKT 5: Wyznaczony Redukt ---\n{chosen_reduct}")


--- PUNKT 5: Wyznaczony Redukt ---
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


### 6. Reguły

In [25]:
rules = train_df_disc.groupby(chosen_reduct)['target'].agg(lambda x: x.value_counts().index[0]).to_dict()

print("\n--- PUNKT 6: Wybrane Reguły Decyzyjne ---")
for cond, dec in rules.items():
    cond_str = " AND ".join([f"({feat}={val})" for feat, val in zip(chosen_reduct, cond)])
    print(f"IF {cond_str} THEN Class = {iris.target_names[dec]}")



--- PUNKT 6: Wybrane Reguły Decyzyjne ---
IF (sepal length (cm)=High) AND (sepal width (cm)=High) AND (petal length (cm)=High) AND (petal width (cm)=High) THEN Class = virginica
IF (sepal length (cm)=High) AND (sepal width (cm)=Medium) AND (petal length (cm)=High) AND (petal width (cm)=High) THEN Class = virginica
IF (sepal length (cm)=High) AND (sepal width (cm)=Medium) AND (petal length (cm)=High) AND (petal width (cm)=Medium) THEN Class = virginica
IF (sepal length (cm)=High) AND (sepal width (cm)=Medium) AND (petal length (cm)=Medium) AND (petal width (cm)=Medium) THEN Class = versicolor
IF (sepal length (cm)=Low) AND (sepal width (cm)=High) AND (petal length (cm)=Low) AND (petal width (cm)=Low) THEN Class = setosa
IF (sepal length (cm)=Low) AND (sepal width (cm)=Low) AND (petal length (cm)=Low) AND (petal width (cm)=Low) THEN Class = setosa
IF (sepal length (cm)=Low) AND (sepal width (cm)=Low) AND (petal length (cm)=Medium) AND (petal width (cm)=High) THEN Class = virginica
IF (s

### 7. Klasyfikacja

In [26]:
def predict(row):
    key = tuple(row[chosen_reduct])
    return rules.get(key, -1) # -1 jeśli reguła nie istnieje

test_df_disc['pred'] = test_df_disc.apply(predict, axis=1)

y_true = test_df_disc['target']
y_pred = test_df_disc['pred']

covered_mask = y_pred != -1
acc = accuracy_score(y_true[covered_mask], y_pred[covered_mask])
coverage = covered_mask.mean()

print(f"\n--- PUNKT 7: Wyniki ---")
print(f"Dokładność (Accuracy) dla znanych reguł: {acc:.2%}")
print(f"Pokrycie zbioru testowego: {coverage:.2%}")


--- PUNKT 7: Wyniki ---
Dokładność (Accuracy) dla znanych reguł: 100.00%
Pokrycie zbioru testowego: 95.56%


## Wnioski

Reguły mogą nie być perfekcyjne ponieważ mogliśmy przez przypadek usunąć potrzebne wartości ze zbioru testowego